# Sesión 9 · Especialización por dominio y recomendación

Este laboratorio compara arquitecturas existentes y construye una siguiente mejor acción auditable. No usa un LLM para calcular ni ordenar.

## 1. ¿Agente, modelo o código?

| Problema | Componente principal | Control |
|---|---|---|
| Fórmula financiera | Código determinista | Fuente, fecha y moneda |
| Interpretación de campaña | Agente + métricas | No afirmar causalidad |
| Acción operativa | Workflow | Permisos y aprobación |
| Priorización | Elegibilidad + scoring | Auditoría por cohortes |

In [ ]:
from pathlib import Path
import pandas as pd
from agents.domain_recommender import CandidateAction, CustomerProfile, RecommendationEngine, RecommendationRequest

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
customers = pd.read_csv(root / 'data' / 'session9' / 'customers.csv')
actions = pd.read_csv(root / 'data' / 'session9' / 'actions.csv')
customers

In [ ]:
def split_set(value):
    return {item for item in str(value).split('|') if item}

profiles = [CustomerProfile(**{**row, 'category_affinity': split_set(row['category_affinity'])}) for row in customers.to_dict('records')]
candidates = [CandidateAction(**{**row, 'eligible_segments': split_set(row['eligible_segments'])}) for row in actions.to_dict('records')]
engine = RecommendationEngine()
result = engine.recommend(RecommendationRequest(customer=profiles[0], candidates=candidates, top_k=3))
pd.DataFrame([item.model_dump() for item in result.recommendations])

## 2. Inspeccionar exclusiones

Una política dura no debe convertirse en una penalización pequeña. Consentimiento, elegibilidad y riesgo máximo se resuelven antes del ranking.

In [ ]:
result.excluded

## 3. Experimentos

1. Aumente el costo de un incentivo.
2. Cambie el canal preferido.
3. Marque `contact_allowed=False`.
4. Reduzca `max_operational_risk`.
5. Discuta qué población podría quedar sistemáticamente excluida.

Registre para cada cambio: resultado, explicación, riesgo y control.

## 4. Proyecto final

Complete `challenges/session9/project_canvas.md`. Elimine cualquier agente que no tenga responsabilidad, entrada, salida y límite propios.